# 00 · Setup: the whole Tinker API on one slide

Three primitives. Every method in this webinar is a different arrangement of them.

| call | what it does |
|---|---|
| `training_client.forward_backward(data, loss_fn)` | compute loss + gradients (on Tinker's GPUs) |
| `training_client.optim_step(adam_params)` | apply the update |
| `sampling_client.sample(prompt, ...)` | generate from current weights |

You write the loop on your laptop; Tinker runs the GPUs.

In [7]:
from common import *

# What can we train?
for m in tinker.RestClient().get_server_capabilities().result().supported_models:
    print(m.model_name)

Qwen/Qwen3-8B
Qwen/Qwen3.5-35B-A3B-Base
Qwen/Qwen3.5-397B-A17B
Qwen/Qwen3.5-397B-A17B:peft:262144
Qwen/Qwen3.5-4B
Qwen/Qwen3.5-4B:peft:262144
Qwen/Qwen3.5-9B
Qwen/Qwen3.5-9B-Base
Qwen/Qwen3.6-35B-A3B
Qwen/Qwen3.6-35B-A3B:peft:262144
Qwen/Qwen3.8-27B
Qwen/Qwen3.8-27B:peft:262144
deepseek-ai/DeepSeek-V4.1-Flash-1017:peft:262144
deepseek-ai/DeepSeek-V4.1-Flash:peft:262144
moonshotai/Kimi-K2.6
moonshotai/Kimi-K2.6:peft:131072
moonshotai/Kimi-K3.1-Pro:peft:262144
nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16
nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16:peft:262144
nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16
nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16:peft:262144
nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16
nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16:peft:262144
nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16
nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16:peft:262144
openai/gpt-oss-120b
openai/gpt-oss-120b:peft:131072
openai/gpt-oss-20b
thinkingmachines/Inkling
thinkin

In [8]:
training_client = new_training_client()
sampling_client = training_client.save_weights_and_get_sampling_client().result()
print("Training + sampling clients ready for", MODEL)

Training + sampling clients ready for Qwen/Qwen3.5-4B


## Baseline: what does the model do before we train it?

In [9]:
show("raw text completion", generate(sampling_client, "To be, or not to be,", chat=False, temperature=0.7))
show("chat prompt", generate(sampling_client, "What's a good name for a pet goldfish?"))
show("math prompt", generate(sampling_client, gsm8k_convo(load_gsm8k("test")[0]["question"]), max_tokens=MAX_TOKENS))


── raw text completion ──
that is the question
As a middle school student in the 1950s, I remember my first English class. The teacher, a young and passionate woman, called upon students to recite the first lines of Shakespeare's "To be, or not to be, that is the question" from "Hamlet". She told us that this is the most famous quote in the English language.
As I grew older, I failed to understand the meaning behind the quote. I simply memorized the words without grasping the deeper significance. Now, as a middle-aged woman, I have realized the profound meaning.
In my


── chat prompt ──
Choosing a name for a goldfish is a fun task because they have such a wide variety of colors, patterns, and personalities! Here are a few ideas categorized by style to help you find the perfect match:

### 🌈 Color & Pattern Inspired
Goldfish are often named after their specific markings or hues.
*   **For bright oranges/red:** Mango, Tangerine, Coral, Fireball, Sunset.
*   **For whites/blacks:** Snowb

In [10]:
# Our task for the day: grade-school math (GSM8K) answered correctly *within 256 tokens*.
# The model knows the math but rambles, so many answers run out of tokens before \boxed{}.
# We'll come back to this score after every section.
eval_gsm8k(sampling_client, label="0. base model")

GSM8K accuracy within 256 tokens [0. base model]: 55.0% (55/100)   [44 ran out of tokens]


0.55